In [1]:
!pip install "numpy" "pandas" "scikit-learn" "openpyxl"
# contoh untuk CUDA 12.1
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install "xgboost"
!pip install os


  Using cached scikit_learn-1.7.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached scipy-1.16.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached scikit_learn-1.7.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.5 MB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached joblib-1.5.2-py3-none-any.whl (308 kB)
Using cached scipy-1.16.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (35.7 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [openpyxl]5/6 [openpyxl]arn]
Looking in indexes: htt

In [3]:
# ============================================================
# LOOP SEMUA HORIZON — UJI MANDIRI per MODEL (tanpa stacking)
# Output: tabel metrik (MAE, MSE, R2) per horizon per model
# ============================================================

from pathlib import Path
import re, warnings, math
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

# XGBoost optional
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ----------------- EDIT DI SINI -----------------
DATA_PATH       = r"/workspace/Machine_Value_Processed.xlsx"
SHEET_NAME      = "SW_3min"   # jika Excel; jika CSV set None di bawah
IS_EXCEL        = True        # True = baca Excel sheet; False = baca CSV
HORIZONS        = ["y_step_3m","y_step_6m","y_step_12m","y_step_24m","y_step_48m"]
MAX_LAG         = 3           # pakai y_lag_1..y_lag_MAX_LAG untuk model klasik
VAL_RATIO       = 0.15
TEST_RATIO      = 0.15

# Deep settings
SEQ_LEN   = 32
BATCH     = 64
EPOCHS    = 30
LR        = 1e-3
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
SEED      = 42

SAVE_CSV  = True
CSV_OUT   = "metrics_all_models_all_horizons.csv"
# ------------------------------------------------

def set_seed(s=42):
    import random
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)

def time_split_idx(n, val_ratio=0.15, test_ratio=0.15):
    n_test = int(n*test_ratio); n_val = int(n*val_ratio)
    n_train = n - n_val - n_test
    assert min(n_train, n_val, n_test) > 0, "Dataset terlalu kecil untuk rasio ini."
    return n_train, n_train+n_val

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    return mae, mse, r2

# ====== LOAD DATA ======
path = Path(DATA_PATH); assert path.exists(), f"Tidak ditemukan: {DATA_PATH}"
if IS_EXCEL:
    df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
else:
    df = pd.read_csv(DATA_PATH)

# pastikan numeric (kecuali id seperti TagNumber)
for c in df.columns:
    if c != "TagNumber":
        df[c] = pd.to_numeric(df[c], errors="ignore")

# siapkan daftar lag tersedia, dan pilih 1..MAX_LAG
lag_cols_all = [c for c in df.columns if re.fullmatch(r"y_lag_\d+", c)]
lag_idx = sorted([int(c.split("_")[-1]) for c in lag_cols_all])
assert lag_idx, "Kolom y_lag_* tidak ditemukan."
use_lags = [f"y_lag_{k}" for k in lag_idx if k <= MAX_LAG]
assert len(use_lags) >= 1, "Kolom lag sesuai MAX_LAG tidak tersedia."

# ====== ARSITEKTUR DEEP ======
class SeqDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class RNNReg(nn.Module):
    def __init__(self, cell="rnn", hid=128, layers=1, dropout=0.1):
        super().__init__()
        in_dim = 1
        if cell=="rnn":
            self.core = nn.RNN(in_dim, hid, num_layers=layers, batch_first=True,
                               nonlinearity="tanh", dropout=dropout if layers>1 else 0.0)
        elif cell=="lstm":
            self.core = nn.LSTM(in_dim, hid, num_layers=layers, batch_first=True,
                                dropout=dropout if layers>1 else 0.0)
        elif cell=="gru":
            self.core = nn.GRU(in_dim, hid, num_layers=layers, batch_first=True,
                               dropout=dropout if layers>1 else 0.0)
        else:
            raise ValueError("cell must be rnn/lstm/gru")
        self.fc = nn.Linear(hid, 1); self.cell = cell
    def forward(self, x):
        out,_ = self.core(x)
        z = out[:, -1, :]
        return self.fc(z).squeeze(-1)

def train_nn(model, tr_dl, va_dl, epochs=EPOCHS, lr=LR):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    loss = nn.MSELoss()
    best = {"mse": float("inf"), "state": None}
    for ep in range(1, epochs+1):
        model.train(); tot = 0.0
        for xb,yb in tr_dl:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); pred = model(xb); l = loss(pred, yb)
            l.backward(); opt.step(); tot += l.item()*len(yb)
        # validasi cepat
        model.eval(); totv = 0.0
        with torch.no_grad():
            for xb,yb in va_dl:
                xb,yb = xb.to(DEVICE), yb.to(DEVICE)
                totv += loss(model(xb), yb).item()*len(yb)
        val_mse = totv/len(va_dl.dataset)
        if val_mse < best["mse"]:
            best["mse"] = val_mse
            best["state"] = {k:v.cpu().clone() for k,v in model.state_dict().items()}
        if ep % 5 == 0 or ep == 1:
            print(f"  Epoch {ep:02d} | val MSE={val_mse:.4f}")
    model.load_state_dict(best["state"])
    return model

def predict_nn(model, dl):
    model.eval(); preds=[]; tgts=[]
    with torch.no_grad():
        for xb,yb in dl:
            pr = model(xb.to(DEVICE)).cpu().numpy()
            preds.append(pr); tgts.append(yb.numpy())
    y_s = np.concatenate(tgts); p_s = np.concatenate(preds)
    return y_s, p_s

def make_seq_from_value_and_target(df_in, target_col, seq_len=32):
    """Bangun dataset sekuens untuk DL:
       Input  = window pada deret Value (skalakan)
       Target = target_col (skalakan)
       Output X: (N, T, 1), y: (N,) dalam ruang ter-skala + scaler target"""
    tmp = df_in[["Value", target_col]].dropna().reset_index(drop=True)
    val = tmp["Value"].astype(float).values
    tgt = tmp[target_col].astype(float).values

    sx = StandardScaler().fit(val.reshape(-1,1))
    sy = StandardScaler().fit(tgt.reshape(-1,1))
    val_s = sx.transform(val.reshape(-1,1)).ravel()
    tgt_s = sy.transform(tgt.reshape(-1,1)).ravel()

    # windowing
    Xs, Ys = [], []
    for i in range(len(tgt_s) - seq_len):
        Xs.append(val_s[i:i+seq_len])
        Ys.append(tgt_s[i+seq_len])
    X = np.array(Xs, dtype=np.float32)[..., None]  # (N, T, 1)
    y = np.array(Ys, dtype=np.float32)
    return X, y, sy

# ====== LOOP SEMUA HORIZON ======
rows = []
for target_col in HORIZONS:
    if target_col not in df.columns:
        print(f"[SKIP] {target_col} tidak ada di data.")
        continue

    print(f"\n================= Horizon: {target_col} =================")

    # ---------- Bagian A: Model klasik (fit ke lag1..MAX_LAG) ----------
    classic_df = df[use_lags + [target_col]].dropna().reset_index(drop=True)
    X = classic_df[use_lags].values.astype(float)
    y = classic_df[target_col].values.astype(float)

    n_train, n_trainval = time_split_idx(len(y), VAL_RATIO, TEST_RATIO)
    X_tr, y_tr = X[:n_train], y[:n_train]
    X_va, y_va = X[n_train:n_trainval], y[n_train:n_trainval]
    X_te, y_te = X[n_trainval:], y[n_trainval:]

    # Linear
    lin = LinearRegression().fit(X_tr, y_tr)
    m = evaluate(y_te, lin.predict(X_te))
    rows.append({"horizon": target_col, "model": "Linear", "MAE": m[0], "MSE": m[1], "R2": m[2]})

    # MLP
    mlp = MLPRegressor(hidden_layer_sizes=(128,64), activation="relu",
                       random_state=SEED, max_iter=800, learning_rate_init=1e-3).fit(X_tr, y_tr)
    m = evaluate(y_te, mlp.predict(X_te))
    rows.append({"horizon": target_col, "model": "MLP", "MAE": m[0], "MSE": m[1], "R2": m[2]})

    # RF
    rf = RandomForestRegressor(n_estimators=400, random_state=SEED, n_jobs=-1).fit(X_tr, y_tr)
    m = evaluate(y_te, rf.predict(X_te))
    rows.append({"horizon": target_col, "model": "RandomForest", "MAE": m[0], "MSE": m[1], "R2": m[2]})

    # XGB / GBDT
    if HAS_XGB:
        xgb = XGBRegressor(n_estimators=600, max_depth=6, learning_rate=0.05,
                           subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
                           random_state=SEED, tree_method="hist").fit(X_tr, y_tr)
        mdl_name = "XGBoost"
    else:
        xgb = GradientBoostingRegressor(random_state=SEED).fit(X_tr, y_tr)
        mdl_name = "GBDT"
    m = evaluate(y_te, xgb.predict(X_te))
    rows.append({"horizon": target_col, "model": mdl_name, "MAE": m[0], "MSE": m[1], "R2": m[2]})

    # ---------- Bagian B: Deep (RNN/LSTM/GRU) ----------
    # Build sekuens dari Value untuk memprediksi target_col
    X_all, y_all, scaler_y = make_seq_from_value_and_target(df, target_col, SEQ_LEN)
    n_train, n_trainval = time_split_idx(len(y_all), VAL_RATIO, TEST_RATIO)
    Xtr_s, ytr_s = X_all[:n_train], y_all[:n_train]
    Xva_s, yva_s = X_all[n_train:n_trainval], y_all[n_train:n_trainval]
    Xte_s, yte_s = X_all[n_trainval:], y_all[n_trainval:]

    tr_dl = DataLoader(SeqDS(Xtr_s, ytr_s), batch_size=BATCH, shuffle=True)
    va_dl = DataLoader(SeqDS(Xva_s, yva_s), batch_size=BATCH, shuffle=False)
    te_dl = DataLoader(SeqDS(Xte_s, yte_s), batch_size=BATCH, shuffle=False)

    # SimpleRNN
    print("  [DL] SimpleRNN")
    rnn = RNNReg(cell="rnn")
    rnn = train_nn(rnn, tr_dl, va_dl, epochs=EPOCHS, lr=LR)
    y_s, p_s = predict_nn(rnn, te_dl)
    y = scaler_y.inverse_transform(y_s.reshape(-1,1)).ravel()
    p = scaler_y.inverse_transform(p_s.reshape(-1,1)).ravel()
    m = evaluate(y, p)
    rows.append({"horizon": target_col, "model": "SimpleRNN", "MAE": m[0], "MSE": m[1], "R2": m[2]})

    # LSTM
    print("  [DL] LSTM")
    lstm = RNNReg(cell="lstm")
    lstm = train_nn(lstm, tr_dl, va_dl, epochs=EPOCHS, lr=LR)
    y_s, p_s = predict_nn(lstm, te_dl)
    y = scaler_y.inverse_transform(y_s.reshape(-1,1)).ravel()
    p = scaler_y.inverse_transform(p_s.reshape(-1,1)).ravel()
    m = evaluate(y, p)
    rows.append({"horizon": target_col, "model": "LSTM", "MAE": m[0], "MSE": m[1], "R2": m[2]})

    # GRU
    print("  [DL] GRU")
    gru = RNNReg(cell="gru")
    gru = train_nn(gru, tr_dl, va_dl, epochs=EPOCHS, lr=LR)
    y_s, p_s = predict_nn(gru, te_dl)
    y = scaler_y.inverse_transform(y_s.reshape(-1,1)).ravel()
    p = scaler_y.inverse_transform(p_s.reshape(-1,1)).ravel()
    m = evaluate(y, p)
    rows.append({"horizon": target_col, "model": "GRU", "MAE": m[0], "MSE": m[1], "R2": m[2]})

# ====== RANGKUM HASIL ======
results = pd.DataFrame(rows).sort_values(["horizon","model"]).reset_index(drop=True)
print("\n===== HASIL RINGKAS =====")
print(results)

if SAVE_CSV:
    results.to_csv(CSV_OUT, index=False)
    print(f"\n✅ Tabel metrik disimpan ke: {CSV_OUT}")


/tmp/ipykernel_432/3736110880.py:78: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_numeric(df[c], errors="ignore")



================= Horizon: y_step_3m =================
  [DL] SimpleRNN
  Epoch 01 | val MSE=0.0002
  Epoch 05 | val MSE=0.0002
  Epoch 10 | val MSE=0.0001
  Epoch 15 | val MSE=0.0002
  Epoch 20 | val MSE=0.0001
  Epoch 25 | val MSE=0.0001
  Epoch 30 | val MSE=0.0002
  [DL] LSTM
  Epoch 01 | val MSE=0.0003
  Epoch 05 | val MSE=0.0002
  Epoch 10 | val MSE=0.0003
  Epoch 15 | val MSE=0.0002
  Epoch 20 | val MSE=0.0001
  Epoch 25 | val MSE=0.0001
  Epoch 30 | val MSE=0.0001
  [DL] GRU
  Epoch 01 | val MSE=0.0003
  Epoch 05 | val MSE=0.0003
  Epoch 10 | val MSE=0.0002
  Epoch 15 | val MSE=0.0002
  Epoch 20 | val MSE=0.0001
  Epoch 25 | val MSE=0.0001
  Epoch 30 | val MSE=0.0001

================= Horizon: y_step_6m =================
  [DL] SimpleRNN
  Epoch 01 | val MSE=0.0003
  Epoch 05 | val MSE=0.0003
  Epoch 10 | val MSE=0.0002
  Epoch 15 | val MSE=0.0003
  Epoch 20 | val MSE=0.0005
  Epoch 25 | val MSE=0.0002
  Epoch 30 | val MSE=0.0002
  [DL] LSTM
  Epoch 01 | val MSE=0.0003
  Epoch

In [6]:
# ============================================================
# PIPELINE: Linear → RF Residual → (Prediksi) → LSTM
# - Loop semua horizon y_step_*
# - LSTM PRETRAIN pakai slot prediksi yang diisi VALUE asli
# - LSTM FINETUNE pakai slot prediksi dari Linear+RF (combo)
# ============================================================

from pathlib import Path
import re, math, warnings
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ----------------- EDIT DI SINI -----------------
DATA_PATH   = r"/workspace/Machine_Value_Processed.xlsx"
SHEET_NAME  = "SW_3min"     # kalau CSV: set IS_EXCEL=False (lihat bawah)
IS_EXCEL    = True
MAX_LAG     = 3             # pakai y_lag_1..y_lag_MAX_LAG
HORIZONS    = ["y_step_3m","y_step_6m","y_step_12m","y_step_24m","y_step_48m"]

VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# LSTM
SEQ_LEN         = 32
BATCH           = 64
EPOCHS_PRETRAIN = 15        # LSTM pretrain dgn slot=VALUE
EPOCHS_FINETUNE = 15        # LSTM finetune dgn slot=prediksi model
LR              = 1e-3
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
SEED            = 42

SAVE_CSV  = True
CSV_OUT   = "metrics_pipeline_lrrf_lstm_all_horizons.csv"
# ------------------------------------------------

def set_seed(s=42):
    import random
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); 
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)

def time_split_idx(n, val_ratio=0.15, test_ratio=0.15):
    n_test = int(n*test_ratio); n_val = int(n*val_ratio)
    n_train = n - n_val - n_test
    assert min(n_train, n_val, n_test) > 0, "Dataset terlalu kecil untuk rasio ini."
    return n_train, n_train+n_val

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    return mae, mse, r2

# ====== LOAD ======
p = Path(DATA_PATH); assert p.exists(), f"Tidak ditemukan: {DATA_PATH}"
df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME) if IS_EXCEL else pd.read_csv(DATA_PATH)

# pastikan numerik
for c in df.columns:
    if c != "TagNumber":
        df[c] = pd.to_numeric(df[c], errors="ignore")

# lag columns
lag_cols_all = [c for c in df.columns if re.fullmatch(r"y_lag_\d+", c)]
lag_idx = sorted([int(c.split("_")[-1]) for c in lag_cols_all])
assert lag_idx, "Kolom y_lag_* tidak ditemukan."
use_lags = [f"y_lag_{k}" for k in lag_idx if k <= MAX_LAG]
assert len(use_lags) >= 1, "Kolom lag sesuai MAX_LAG tidak tersedia."

# ====== Dataset & Model LSTM ======
class SeqDS(Dataset):
    def __init__(self, X, y): self.X=torch.from_numpy(X); self.y=torch.from_numpy(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class LSTMReg(nn.Module):
    def __init__(self, in_dim=6, hid=128, layers=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hid, num_layers=layers, batch_first=True,
                            dropout=dropout if layers>1 else 0.0)
        self.fc   = nn.Linear(hid, 1)
    def forward(self, x):
        out,_ = self.lstm(x)
        z = out[:, -1, :]
        return self.fc(z).squeeze(-1)

def train_nn(model, tr_dl, va_dl, epochs=10, lr=1e-3):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    loss = nn.MSELoss()
    best = {"mse": float("inf"), "state": None}
    for ep in range(1, epochs+1):
        model.train(); tot=0.0
        for xb,yb in tr_dl:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); pred = model(xb); l=loss(pred,yb)
            l.backward(); opt.step(); tot += l.item()*len(yb)
        # val
        model.eval(); totv=0.0
        with torch.no_grad():
            for xb,yb in va_dl:
                xb,yb = xb.to(DEVICE), yb.to(DEVICE)
                totv += loss(model(xb), yb).item()*len(yb)
        val_mse = totv/len(va_dl.dataset)
        if val_mse < best["mse"]:
            best["mse"] = val_mse
            best["state"] = {k:v.cpu().clone() for k,v in model.state_dict().items()}
        if ep % 5 == 0 or ep==1:
            print(f"  Epoch {ep:02d} | val MSE={val_mse:.4f}")
    model.load_state_dict(best["state"]); 
    return model

def predict_nn(model, dl):
    model.eval(); preds=[]; tgts=[]
    with torch.no_grad():
        for xb,yb in dl:
            pr = model(xb.to(DEVICE)).cpu().numpy()
            preds.append(pr); tgts.append(yb.numpy())
    return np.concatenate(tgts), np.concatenate(preds)

def build_classic_mats(df, lag_cols, target_col):
    d = df[lag_cols + [target_col]].dropna().reset_index(drop=True)
    X = d[lag_cols].values.astype(float)
    y = d[target_col].values.astype(float)
    return X, y, d.index.values  # kembalikan index mapping

def build_seq_with_slots(df, lag_cols, target_col, lin_full, combo_full, idx_map,
                         seq_len=32, use_pred_slots=False):
    """
    Buat sekuens LSTM dengan channel: [lag1, lag2, lag3, Value, lin_slot, rf_slot]
    - Saat PRETRAIN: use_pred_slots=False → isi lin_slot & rf_slot = VALUE
    - Saat FINETUNE: use_pred_slots=True  → isi dari lin_full & combo_full (sesuai index)
    Align:
      - Hanya baris yang memiliki semua kolom dan masuk mapping idx_map (agar sinkron).
    """
    cols_needed = ["Value"] + lag_cols + [target_col]
    tmp = df[cols_needed].dropna().reset_index()
    # ambil yang juga ada di classic (idx_map)
    tmp = tmp[tmp["index"].isin(idx_map)].reset_index(drop=True)

    val = tmp["Value"].astype(float).values
    lags = tmp[lag_cols].values.astype(float)
    tgt  = tmp[target_col].astype(float).values
    # scaler
    sx = StandardScaler().fit(np.column_stack([lags, val]))
    sy = StandardScaler().fit(tgt.reshape(-1,1))
    base = sx.transform(np.column_stack([lags, val]))
    l1, l2, l3 = base[:,0], base[:,1], base[:,2]
    v_norm     = base[:,3]

    # siapkan slot
    if use_pred_slots:
        # mapping lin_full/combo_full ke baris tmp (berdasarkan posisi idx di idx_map)
        # Buat dict: global_index -> position di classic
        pos = {idx:i for i,idx in enumerate(idx_map)}
        lin_aligned   = np.array([lin_full[pos[i]] for i in tmp["index"].values], dtype=float)
        combo_aligned = np.array([combo_full[pos[i]] for i in tmp["index"].values], dtype=float)
        # scale ke target space:
        lin_s   = sy.transform(lin_aligned.reshape(-1,1)).ravel()
        combo_s = sy.transform(combo_aligned.reshape(-1,1)).ravel()
        slot1, slot2 = lin_s, combo_s
    else:
        # isi dengan value asli (di ruang target!) — konsisten dengan target scaler
        # konversi value ke skala target agar scale seragam
        # gunakan proporsional terhadap target: scale VALUE menggunakan sy.fit pada tgt
        slot_val = sy.transform(val.reshape(-1,1)).ravel()
        slot1, slot2 = slot_val, slot_val

    # target ter-scale
    y_s = sy.transform(tgt.reshape(-1,1)).ravel()

    # rakit fitur per-waktu: [lag1, lag2, lag3, value_norm, slot1, slot2]
    F = np.column_stack([l1, l2, l3, v_norm, slot1, slot2]).astype(np.float32)

    # windowing
    Xs, Ys = [], []
    for i in range(len(y_s) - seq_len):
        Xs.append(F[i:i+seq_len])
        Ys.append(y_s[i+seq_len])
    X = np.array(Xs, dtype=np.float32)   # (N, T, C)
    y = np.array(Ys, dtype=np.float32)
    return X, y, sy

# ====== MAIN LOOP ======
rows = []

for target_col in HORIZONS:
    if target_col not in df.columns:
        print(f"[SKIP] {target_col} tidak ada.")
        continue
    print(f"\n================= Horizon: {target_col} =================")

    # ---------- 1) Linear + RF residual (fit di lag) ----------
    X_all, y_all, idx_map = build_classic_mats(df, use_lags, target_col)
    n_train, n_trainval = time_split_idx(len(y_all), VAL_RATIO, TEST_RATIO)
    X_tr, y_tr = X_all[:n_train], y_all[:n_train]
    X_va, y_va = X_all[n_train:n_trainval], y_all[n_train:n_trainval]
    X_te, y_te = X_all[n_trainval:], y_all[n_trainval:]

    lin = LinearRegression().fit(X_tr, y_tr)
    yhat_lin_tr = lin.predict(X_tr)
    yhat_lin_va = lin.predict(X_va)
    yhat_lin_te = lin.predict(X_te)

    # residual
    res_tr = y_tr - yhat_lin_tr
    rf = RandomForestRegressor(n_estimators=400, random_state=SEED, n_jobs=-1).fit(X_tr, res_tr)
    rhat_va = rf.predict(X_va); rhat_te = rf.predict(X_te)

    # gabungan
    combo_va = yhat_lin_va + rhat_va
    combo_te = yhat_lin_te + rhat_te

    # gabungan utk seluruh indeks (untuk FINETUNE LSTM)
    # kita prediksi full (train+val+test) agar bisa disejajarkan dengan sekuens
    rhat_tr  = rf.predict(X_tr)
    combo_tr = yhat_lin_tr + rhat_tr
    lin_full   = np.concatenate([yhat_lin_tr, yhat_lin_va, yhat_lin_te])
    combo_full = np.concatenate([combo_tr, combo_va, combo_te])

    # ---------- 2) LSTM PRETRAIN (slot=VALUE asli) ----------
    X_s, y_s, sy = build_seq_with_slots(
        df, use_lags, target_col,
        lin_full=None, combo_full=None, idx_map=idx_map,
        seq_len=SEQ_LEN, use_pred_slots=False
    )
    n_train, n_trainval = time_split_idx(len(y_s), VAL_RATIO, TEST_RATIO)
    Xtr_s, ytr_s = X_s[:n_train], y_s[:n_train]
    Xva_s, yva_s = X_s[n_train:n_trainval], y_s[n_train:n_trainval]
    Xte_s, yte_s = X_s[n_trainval:], y_s[n_trainval:]

    tr_dl = DataLoader(SeqDS(Xtr_s, ytr_s), batch_size=BATCH, shuffle=True)
    va_dl = DataLoader(SeqDS(Xva_s, yva_s), batch_size=BATCH, shuffle=False)
    te_dl = DataLoader(SeqDS(Xte_s, yte_s), batch_size=BATCH, shuffle=False)

    print("  [LSTM] PRETRAIN (slot=VALUE)")
    lstm = LSTMReg(in_dim=6, hid=128, layers=1, dropout=0.1)
    lstm = train_nn(lstm, tr_dl, va_dl, epochs=EPOCHS_PRETRAIN, lr=LR)

    # ---------- 3) LSTM FINETUNE (slot=prediksi Linear+RF) ----------
    X_s2, y_s2, sy2 = build_seq_with_slots(
        df, use_lags, target_col,
        lin_full=lin_full, combo_full=combo_full, idx_map=idx_map,
        seq_len=SEQ_LEN, use_pred_slots=True
    )
    # gunakan split yang sama proporsinya
    n_train2, n_trainval2 = time_split_idx(len(y_s2), VAL_RATIO, TEST_RATIO)
    Xtr2, ytr2 = X_s2[:n_train2], y_s2[:n_train2]
    Xva2, yva2 = X_s2[n_train2:n_trainval2], y_s2[n_train2:n_trainval2]
    Xte2, yte2 = X_s2[n_trainval2:], y_s2[n_trainval2:]

    tr2 = DataLoader(SeqDS(Xtr2, ytr2), batch_size=BATCH, shuffle=True)
    va2 = DataLoader(SeqDS(Xva2, yva2), batch_size=BATCH, shuffle=False)
    te2 = DataLoader(SeqDS(Xte2, yte2), batch_size=BATCH, shuffle=False)

    print("  [LSTM] FINETUNE (slot=Prediksi Linear+RF)")
    lstm = train_nn(lstm, tr2, va2, epochs=EPOCHS_FINETUNE, lr=LR)  # lanjut dari bobot pretrain

    # ---------- 4) Evaluasi ----------
    # a) evaluasi linear/RF di test klasik
    mae, mse, r2 = evaluate(y_te, yhat_lin_te); 
    rows.append({"horizon": target_col, "stage": "Linear", "MAE": mae, "MSE": mse, "R2": r2})
    mae, mse, r2 = evaluate(y_te, combo_te);     
    rows.append({"horizon": target_col, "stage": "Linear+RFres", "MAE": mae, "MSE": mse, "R2": r2})

    # b) evaluasi LSTM (pakai set FINETUNE test)
    y_s_true, y_s_pred = predict_nn(lstm, te2)
    # inverse transform ke satuan target asli
    y_true = sy2.inverse_transform(y_s_true.reshape(-1,1)).ravel()
    y_pred = sy2.inverse_transform(y_s_pred.reshape(-1,1)).ravel()
    mae, mse, r2 = evaluate(y_true, y_pred)
    rows.append({"horizon": target_col, "stage": "LSTM(finetuned)", "MAE": mae, "MSE": mse, "R2": r2})

# ====== OUTPUT RINGKAS ======
results = pd.DataFrame(rows).sort_values(["horizon","stage"]).reset_index(drop=True)
print("\n===== HASIL RINGKAS =====")
print(results)

if SAVE_CSV:
    results.to_csv(CSV_OUT, index=False)
    print(f"\n✅ Tabel metrik disimpan ke: {CSV_OUT}")


/tmp/ipykernel_432/1425544610.py:71: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_numeric(df[c], errors="ignore")



================= Horizon: y_step_3m =================
  [LSTM] PRETRAIN (slot=VALUE)
  Epoch 01 | val MSE=0.0003
  Epoch 05 | val MSE=0.0002
  Epoch 10 | val MSE=0.0003
  Epoch 15 | val MSE=0.0001
  [LSTM] FINETUNE (slot=Prediksi Linear+RF)
  Epoch 01 | val MSE=0.0002
  Epoch 05 | val MSE=0.0002
  Epoch 10 | val MSE=0.0001
  Epoch 15 | val MSE=0.0001

================= Horizon: y_step_6m =================
  [LSTM] PRETRAIN (slot=VALUE)
  Epoch 01 | val MSE=0.0003
  Epoch 05 | val MSE=0.0003
  Epoch 10 | val MSE=0.0002
  Epoch 15 | val MSE=0.0003
  [LSTM] FINETUNE (slot=Prediksi Linear+RF)
  Epoch 01 | val MSE=0.0002
  Epoch 05 | val MSE=0.0003
  Epoch 10 | val MSE=0.0002
  Epoch 15 | val MSE=0.0003

================= Horizon: y_step_12m =================
  [LSTM] PRETRAIN (slot=VALUE)
  Epoch 01 | val MSE=0.0004
  Epoch 05 | val MSE=0.0004
  Epoch 10 | val MSE=0.0003
  Epoch 15 | val MSE=0.0003
  [LSTM] FINETUNE (slot=Prediksi Linear+RF)
  Epoch 01 | val MSE=0.0003
  Epoch 05 | val M

In [7]:
# ============================================================
# LRRf → replace VALUE with (Linear+RF) predictions → LSTM
# - Loop semua horizon y_step_*
# - VALUE di channel sekuens diganti oleh prediksi combo (lin+RF)
# - Slot tambahan: lin_pred & combo_pred
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ----------------- EDIT DI SINI -----------------
DATA_PATH   = r"/workspace/Machine_Value_Processed.xlsx"
SHEET_NAME  = "SW_3min"   # jika CSV: set IS_EXCEL=False
IS_EXCEL    = True

HORIZONS    = ["y_step_3m","y_step_6m","y_step_12m","y_step_24m","y_step_48m"]
MAX_LAG     = 3
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# LSTM
SEQ_LEN   = 32
BATCH     = 64
EPOCHS    = 25
LR        = 1e-3
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
SEED      = 42

SAVE_CSV  = True
CSV_OUT   = "metrics_replaceValue_combo_LSTM.csv"
# ------------------------------------------------

def set_seed(s=42):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)

def time_split_idx(n, val_ratio=0.15, test_ratio=0.15):
    n_test = int(n*test_ratio); n_val = int(n*val_ratio)
    n_train = n - n_val - n_test
    assert min(n_train, n_val, n_test) > 0, "Dataset terlalu kecil."
    return n_train, n_train+n_val

def evaluate(y_true, y_pred):
    return (mean_absolute_error(y_true, y_pred),
            mean_squared_error(y_true, y_pred),
            r2_score(y_true, y_pred))

# ===== LOAD =====
p = Path(DATA_PATH); assert p.exists(), f"Tidak ditemukan: {DATA_PATH}"
df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME) if IS_EXCEL else pd.read_csv(DATA_PATH)
for c in df.columns:
    if c != "TagNumber":
        df[c] = pd.to_numeric(df[c], errors="ignore")

# lag columns
lag_cols_all = [c for c in df.columns if re.fullmatch(r"y_lag_\d+", c)]
lag_idx = sorted(int(c.split("_")[-1]) for c in lag_cols_all)
assert lag_idx, "Kolom y_lag_* tidak ditemukan."
use_lags = [f"y_lag_{k}" for k in lag_idx if k <= MAX_LAG]
assert len(use_lags) >= 1, "Kolom lag sesuai MAX_LAG tidak tersedia."

# ===== Datasets & Model =====
class SeqDS(Dataset):
    def __init__(self, X, y): self.X=torch.from_numpy(X); self.y=torch.from_numpy(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class LSTMReg(nn.Module):
    def __init__(self, in_dim=6, hid=128, layers=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hid, num_layers=layers, batch_first=True,
                            dropout=dropout if layers>1 else 0.0)
        self.fc   = nn.Linear(hid, 1)
    def forward(self, x):
        out,_ = self.lstm(x); z = out[:, -1, :]
        return self.fc(z).squeeze(-1)

def train_nn(model, tr_dl, va_dl, epochs=10, lr=1e-3):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    loss = nn.MSELoss()
    best = {"mse": float("inf"), "state": None}
    for ep in range(1, epochs+1):
        model.train(); tot=0.0
        for xb,yb in tr_dl:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); pred = model(xb); l=loss(pred,yb)
            l.backward(); opt.step(); tot += l.item()*len(yb)
        model.eval(); totv=0.0
        with torch.no_grad():
            for xb,yb in va_dl:
                xb,yb = xb.to(DEVICE), yb.to(DEVICE)
                totv += loss(model(xb), yb).item()*len(yb)
        val_mse = totv/len(va_dl.dataset)
        if val_mse < best["mse"]:
            best["mse"]=val_mse; best["state"]={k:v.cpu().clone() for k,v in model.state_dict().items()}
        if ep % 5 == 0 or ep == 1:
            print(f"  Epoch {ep:02d} | val MSE={val_mse:.4f}")
    model.load_state_dict(best["state"]); 
    return model

def predict_nn(model, dl):
    model.eval(); preds=[]; tgts=[]
    with torch.no_grad():
        for xb,yb in dl:
            pr = model(xb.to(DEVICE)).cpu().numpy()
            preds.append(pr); tgts.append(yb.numpy())
    return np.concatenate(tgts), np.concatenate(preds)

def classic_fit_predict_full(df, lag_cols, target_col, val_ratio, test_ratio):
    """Fit Linear + RF residual di fitur lag; kembalikan prediksi (linear & combo) untuk seluruh baris yang valid."""
    d = df[lag_cols + [target_col]].dropna().reset_index(drop=True)
    X = d[lag_cols].values.astype(float)
    y = d[target_col].values.astype(float)
    n_train, n_trainval = time_split_idx(len(y), val_ratio, test_ratio)
    X_tr, y_tr = X[:n_train], y[:n_train]
    X_va, y_va = X[n_train:n_trainval], y[n_train:n_trainval]
    X_te, y_te = X[n_trainval:], y[n_trainval:]

    lin = LinearRegression().fit(X_tr, y_tr)
    yhat_tr = lin.predict(X_tr); yhat_va = lin.predict(X_va); yhat_te = lin.predict(X_te)
    res_tr  = y_tr - yhat_tr
    rf = RandomForestRegressor(n_estimators=400, random_state=SEED, n_jobs=-1).fit(X_tr, res_tr)
    rhat_tr = rf.predict(X_tr); rhat_va = rf.predict(X_va); rhat_te = rf.predict(X_te)

    lin_full   = np.concatenate([yhat_tr, yhat_va, yhat_te])
    combo_full = np.concatenate([yhat_tr + rhat_tr, yhat_va + rhat_va, yhat_te + rhat_te])
    return d, lin_full, combo_full, (y_te, yhat_te, yhat_te + rhat_te)

def build_seq_replace_value(df, classic_df, lag_cols, target_col, lin_full, combo_full, seq_len=32):
    """
    Bangun sekuens untuk LSTM:
      channels = [lag1, lag2, lag3, combo_as_value, lin_pred, combo_pred]
      - combo_as_value: channel 'Value' diganti oleh prediksi combo (diskalakan)
      - lin_pred & combo_pred: juga ikut sebagai channel
    """
    # align ke baris yang dipakai classic_df
    tmp = df[["Value"] + lag_cols + [target_col]].dropna().reset_index()
    # classic_df punya index lokal 0..N-1; kita pakai panjangnya
    N = len(classic_df)
    tmp = tmp.iloc[:N].copy().reset_index(drop=True)

    # Ambil array mentah
    val = tmp["Value"].astype(float).values
    lags = tmp[lag_cols].values.astype(float)
    tgt  = tmp[target_col].astype(float).values

    # scaler fitur dasar (lags + "value_replaced")
    # kita akan ganti value dengan combo -> tapi normalisasi tetap berdasarkan (lags, value asli) agar stabil
    sx = StandardScaler().fit(np.column_stack([lags, val]))
    sy = StandardScaler().fit(tgt.reshape(-1,1))

    # siapkan channel
    base = sx.transform(np.column_stack([lags, val]))
    l1, l2, l3 = base[:,0], base[:,1], base[:,2]

    # scale lin & combo ke ruang target
    lin_s   = sy.transform(lin_full.reshape(-1,1)).ravel()
    combo_s = sy.transform(combo_full.reshape(-1,1)).ravel()

    # channel 'Value' diganti oleh combo_s, bukan v_norm
    combo_as_value = combo_s.copy()

    # target ter-scale
    y_s = sy.transform(tgt.reshape(-1,1)).ravel()

    # rakit fitur per waktu
    F = np.column_stack([l1, l2, l3, combo_as_value, lin_s, combo_s]).astype(np.float32)

    # windowing
    Xs, Ys = [], []
    for i in range(len(y_s) - seq_len):
        Xs.append(F[i:i+seq_len])
        Ys.append(y_s[i+seq_len])
    X = np.array(Xs, dtype=np.float32)
    y = np.array(Ys, dtype=np.float32)
    return X, y, sy

# ===== MAIN LOOP =====
rows = []
for target_col in HORIZONS:
    if target_col not in df.columns:
        print(f"[SKIP] {target_col} tidak ada."); continue
    print(f"\n================= Horizon: {target_col} =================")

    # 1) Fit Linear + RF residual, dan ambil prediksi FULL
    classic_df, lin_full, combo_full, (y_te, yhat_lin_te, yhat_combo_te) = classic_fit_predict_full(
        df, use_lags, target_col, VAL_RATIO, TEST_RATIO
    )

    # 2) Bangun sekuens dengan 'Value' diganti combo
    X_all, y_all, scaler_y = build_seq_replace_value(
        df, classic_df, use_lags, target_col, lin_full, combo_full, SEQ_LEN
    )

    # 3) Split sekuens, DataLoaders
    n_train, n_trainval = time_split_idx(len(y_all), VAL_RATIO, TEST_RATIO)
    Xtr, ytr = X_all[:n_train], y_all[:n_train]
    Xva, yva = X_all[n_train:n_trainval], y_all[n_train:n_trainval]
    Xte, yte = X_all[n_trainval:], y_all[n_trainval:]

    tr_dl = DataLoader(SeqDS(Xtr, ytr), batch_size=BATCH, shuffle=True)
    va_dl = DataLoader(SeqDS(Xva, yva), batch_size=BATCH, shuffle=False)
    te_dl = DataLoader(SeqDS(Xte, yte), batch_size=BATCH, shuffle=False)

    # 4) Train LSTM (satu tahap)
    print("  [LSTM] Train with VALUE replaced by combo(pred)")
    lstm = LSTMReg(in_dim=6, hid=128, layers=1, dropout=0.1)
    lstm = train_nn(lstm, tr_dl, va_dl, epochs=EPOCHS, lr=LR)

    # 5) Evaluasi LSTM
    y_s_true, y_s_pred = predict_nn(lstm, te_dl)
    y_true = scaler_y.inverse_transform(y_s_true.reshape(-1,1)).ravel()
    y_pred = scaler_y.inverse_transform(y_s_pred.reshape(-1,1)).ravel()
    mae, mse, r2 = evaluate(y_true, y_pred)
    rows.append({"horizon": target_col, "model": "LSTM(Value=combo, +lin/combo slots)", "MAE": mae, "MSE": mse, "R2": r2})

    # (opsional) cetak juga performa baseline klasik di horizon yg sama
    mae, mse, r2 = evaluate(y_te, yhat_lin_te)
    rows.append({"horizon": target_col, "model": "Linear baseline", "MAE": mae, "MSE": mse, "R2": r2})
    mae, mse, r2 = evaluate(y_te, yhat_combo_te)
    rows.append({"horizon": target_col, "model": "Linear+RF residual", "MAE": mae, "MSE": mse, "R2": r2})

# ===== RINGKAS HASIL =====
results = pd.DataFrame(rows).sort_values(["horizon","model"]).reset_index(drop=True)
print("\n===== HASIL RINGKAS =====")
print(results)

if SAVE_CSV:
    results.to_csv(CSV_OUT, index=False)
    print(f"\n✅ Tabel metrik disimpan ke: {CSV_OUT}")


/tmp/ipykernel_432/3438618312.py:66: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_numeric(df[c], errors="ignore")



================= Horizon: y_step_3m =================
  [LSTM] Train with VALUE replaced by combo(pred)
  Epoch 01 | val MSE=0.0003
  Epoch 05 | val MSE=0.0002
  Epoch 10 | val MSE=0.0003
  Epoch 15 | val MSE=0.0002
  Epoch 20 | val MSE=0.0002
  Epoch 25 | val MSE=0.0002

================= Horizon: y_step_6m =================
  [LSTM] Train with VALUE replaced by combo(pred)
  Epoch 01 | val MSE=0.0003
  Epoch 05 | val MSE=0.0003
  Epoch 10 | val MSE=0.0002
  Epoch 15 | val MSE=0.0003
  Epoch 20 | val MSE=0.0003
  Epoch 25 | val MSE=0.0003

================= Horizon: y_step_12m =================
  [LSTM] Train with VALUE replaced by combo(pred)
  Epoch 01 | val MSE=0.0004
  Epoch 05 | val MSE=0.0004
  Epoch 10 | val MSE=0.0003
  Epoch 15 | val MSE=0.0004
  Epoch 20 | val MSE=0.0003
  Epoch 25 | val MSE=0.0003

================= Horizon: y_step_24m =================
  [LSTM] Train with VALUE replaced by combo(pred)
  Epoch 01 | val MSE=0.0004
  Epoch 05 | val MSE=0.0004
  Epoch 10 | v

In [16]:
# ============================================================
# GRU pipeline (3 tahap): PRETRAIN(Value) -> FINETUNE(Combo) -> RE-ANCHOR(Value)
# + Evaluasi akhir dua versi input: VALUE & COMBO (setelah re-anchor)
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ----------------- EDIT DI SINI -----------------
DATA_PATH   = r"/workspace/Machine_Value_Processed.xlsx"
SHEET_NAME  = "SW_3min"
IS_EXCEL    = True

HORIZONS    = ["y_step_3m","y_step_6m","y_step_12m","y_step_24m","y_step_48m"]
MAX_LAG     = 3
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# GRU & training
SEQ_LEN          = 32
BATCH            = 64
EPOCHS_PRETRAIN  = 15
EPOCHS_FINETUNE  = 15
EPOCHS_REANCHOR  = 10
LR               = 1e-3
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
SEED             = 42

SAVE_CSV  = True
CSV_OUT   = "metrics_gru_pretrain_combo_reanchor.csv"
# ------------------------------------------------

def set_seed(s=42):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)

def time_split_idx(n, val_ratio=0.15, test_ratio=0.15):
    n_test = int(n*test_ratio); n_val = int(n*val_ratio)
    n_train = n - n_val - n_test
    assert min(n_train, n_val, n_test) > 0, "Dataset terlalu kecil."
    return n_train, n_train+n_val

def evaluate(y_true, y_pred):
    return (mean_absolute_error(y_true, y_pred),
            mean_squared_error(y_true, y_pred),
            r2_score(y_true, y_pred))

# ===== LOAD =====
p = Path(DATA_PATH); assert p.exists(), f"Tidak ditemukan: {DATA_PATH}"
df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME) if IS_EXCEL else pd.read_csv(DATA_PATH)
for c in df.columns:
    if c != "TagNumber":
        df[c] = pd.to_numeric(df[c], errors="ignore")

# lag columns
lag_cols_all = [c for c in df.columns if re.fullmatch(r"y_lag_\d+", c)]
lag_idx = sorted(int(c.split("_")[-1]) for c in lag_cols_all)
assert lag_idx, "Kolom y_lag_* tidak ditemukan."
use_lags = [f"y_lag_{k}" for k in lag_idx if k <= MAX_LAG]
assert len(use_lags) >= 1, "Kolom lag sesuai MAX_LAG tidak tersedia."

# ===== Datasets & Model =====
class SeqDS(Dataset):
    def __init__(self, X, y): self.X=torch.from_numpy(X); self.y=torch.from_numpy(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class GRUReg(nn.Module):
    def __init__(self, in_dim=6, hid=128, layers=2, dropout=0.1):
        super().__init__()
        self.gru = nn.GRU(in_dim, hid, num_layers=layers, batch_first=True,
                          dropout=dropout if layers>1 else 0.0)
        self.fc  = nn.Linear(hid, 1)
    def forward(self, x):
        out,_ = self.gru(x); z = out[:, -1, :]
        return self.fc(z).squeeze(-1)

def train_nn(model, tr_dl, va_dl, epochs=10, lr=1e-3, name="GRU"):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    loss = nn.MSELoss()
    best = {"mse": float("inf"), "state": None}
    for ep in range(1, epochs+1):
        model.train(); tot=0.0
        for xb,yb in tr_dl:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); pred = model(xb); l=loss(pred,yb)
            l.backward(); opt.step(); tot += l.item()*len(yb)
        # val
        model.eval(); totv=0.0
        with torch.no_grad():
            for xb,yb in va_dl:
                xb,yb = xb.to(DEVICE), yb.to(DEVICE)
                totv += loss(model(xb), yb).item()*len(yb)
        val_mse = totv/len(va_dl.dataset)
        if val_mse < best["mse"]:
            best["mse"]=val_mse; best["state"]={k:v.cpu().clone() for k,v in model.state_dict().items()}
        if ep % 5 == 0 or ep == 1:
            print(f"[{name}] Epoch {ep:02d} | val MSE={val_mse:.4f}")
    model.load_state_dict(best["state"])
    return model

def predict_nn(model, dl):
    model.eval(); preds=[]; tgts=[]
    with torch.no_grad():
        for xb,yb in dl:
            pr = model(xb.to(DEVICE)).cpu().numpy()
            preds.append(pr); tgts.append(yb.numpy())
    return np.concatenate(tgts), np.concatenate(preds)

# ---------- Baseline klasik: Linear + RF residual ----------
def classic_fit_predict_full(df, lag_cols, target_col, val_ratio, test_ratio):
    d = df[lag_cols + [target_col]].dropna().reset_index(drop=True)
    X = d[lag_cols].values.astype(float)
    y = d[target_col].values.astype(float)
    n_train, n_trainval = time_split_idx(len(y), val_ratio, test_ratio)
    X_tr, y_tr = X[:n_train], y[:n_train]
    X_va, y_va = X[n_train:n_trainval], y[n_train:n_trainval]
    X_te, y_te = X[n_trainval:], y[n_trainval:]

    lin = LinearRegression().fit(X_tr, y_tr)
    yhat_tr = lin.predict(X_tr); yhat_va = lin.predict(X_va); yhat_te = lin.predict(X_te)
    res_tr  = y_tr - yhat_tr
    rf = RandomForestRegressor(n_estimators=400, random_state=SEED, n_jobs=-1).fit(X_tr, res_tr)
    rhat_tr = rf.predict(X_tr); rhat_va = rf.predict(X_va); rhat_te = rf.predict(X_te)

    lin_full   = np.concatenate([yhat_tr, yhat_va, yhat_te])
    combo_full = np.concatenate([yhat_tr + rhat_tr, yhat_va + rhat_va, yhat_te + rhat_te])
    return d, lin_full, combo_full, (y_te, yhat_lin_te, yhat_combo_te := yhat_te + rhat_te)

# ---------- Builder sekuens ----------
def build_seq_VALUE(df, classic_df, lag_cols, target_col, seq_len=32):
    # channels = [lag1, lag2, lag3, value_norm, value_slot, value_slot]
    tmp = df[["Value"] + lag_cols + [target_col]].dropna().reset_index()
    N = len(classic_df); tmp = tmp.iloc[:N].copy().reset_index(drop=True)

    val = tmp["Value"].astype(float).values
    lags = tmp[lag_cols].values.astype(float)
    tgt  = tmp[target_col].astype(float).values

    sx = StandardScaler().fit(np.column_stack([lags, val]))
    sy = StandardScaler().fit(tgt.reshape(-1,1))
    base = sx.transform(np.column_stack([lags, val]))
    l1, l2, l3 = base[:,0], base[:,1], base[:,2]
    v_norm     = base[:,3]
    v_slot     = sy.transform(val.reshape(-1,1)).ravel()

    y_s = sy.transform(tgt.reshape(-1,1)).ravel()
    F = np.column_stack([l1, l2, l3, v_norm, v_slot, v_slot]).astype(np.float32)

    Xs, Ys = [], []
    for i in range(len(y_s) - seq_len):
        Xs.append(F[i:i+seq_len]); Ys.append(y_s[i+seq_len])
    return np.array(Xs, np.float32), np.array(Ys, np.float32), sy

def build_seq_COMBO(df, classic_df, lag_cols, target_col, lin_full, combo_full, seq_len=32):
    # channels = [lag1, lag2, lag3, combo_as_value, lin_pred, combo_pred]
    tmp = df[["Value"] + lag_cols + [target_col]].dropna().reset_index()
    N = len(classic_df); tmp = tmp.iloc[:N].copy().reset_index(drop=True)

    val = tmp["Value"].astype(float).values
    lags = tmp[lag_cols].values.astype(float)
    tgt  = tmp[target_col].astype(float).values

    sx = StandardScaler().fit(np.column_stack([lags, val]))
    sy = StandardScaler().fit(tgt.reshape(-1,1))
    base = sx.transform(np.column_stack([lags, val]))
    l1, l2, l3 = base[:,0], base[:,1], base[:,2]

    lin_s   = sy.transform(lin_full.reshape(-1,1)).ravel()
    combo_s = sy.transform(combo_full.reshape(-1,1)).ravel()

    y_s = sy.transform(tgt.reshape(-1,1)).ravel()
    F = np.column_stack([l1, l2, l3, combo_s, lin_s, combo_s]).astype(np.float32)

    Xs, Ys = [], []
    for i in range(len(y_s) - seq_len):
        Xs.append(F[i:i+seq_len]); Ys.append(y_s[i+seq_len])
    return np.array(Xs, np.float32), np.array(Ys, np.float32), sy

# ===== MAIN LOOP =====
rows = []

for target_col in HORIZONS:
    if target_col not in df.columns:
        print(f"[SKIP] {target_col} tidak ada."); continue
    print(f"\n================= Horizon: {target_col} =================")

    # 0) Baseline klasik
    classic_df, lin_full, combo_full, (y_te_cls, yhat_lin_te, yhat_combo_te) = classic_fit_predict_full(
        df, use_lags, target_col, VAL_RATIO, TEST_RATIO
    )

    # helper split
    def make_loaders(X, y):
        n_tr, n_trva = time_split_idx(len(y), VAL_RATIO, TEST_RATIO)
        Xtr, ytr = X[:n_tr], y[:n_tr]
        Xva, yva = X[n_tr:n_trva], y[n_tr:n_trva]
        Xte, yte = X[n_trva:], y[n_trva:]
        tr = DataLoader(SeqDS(Xtr, ytr), batch_size=BATCH, shuffle=True)
        va = DataLoader(SeqDS(Xva, yva), batch_size=BATCH, shuffle=False)
        te = DataLoader(SeqDS(Xte, yte), batch_size=BATCH, shuffle=False)
        return (tr, va, te)

    # 1) PRETRAIN (VALUE)
    X_val_pre, y_val_pre, sy_pre = build_seq_VALUE(df, classic_df, use_lags, target_col, SEQ_LEN)
    tr_dl, va_dl, te_dl = make_loaders(X_val_pre, y_val_pre)
    print("  [GRU] PRETRAIN (Value-as-Value, slots=Value)")
    gru = GRUReg(in_dim=6, hid=128, layers=1, dropout=0.5)
    gru = train_nn(gru, tr_dl, va_dl, epochs=EPOCHS_PRETRAIN, lr=LR, name="GRU-PRE")

    # 2) FINETUNE (COMBO)
    X_combo, y_combo, sy_combo = build_seq_COMBO(df, classic_df, use_lags, target_col, lin_full, combo_full, SEQ_LEN)
    tr2, va2, te2 = make_loaders(X_combo, y_combo)
    print("  [GRU] FINETUNE (Value=Combo, slots=lin/combo)")
    gru = train_nn(gru, tr2, va2, epochs=EPOCHS_FINETUNE, lr=LR, name="GRU-FT-COMBO")

    # 3) RE-ANCHOR (VALUE lagi)
    tr3, va3, te3 = make_loaders(X_val_pre, y_val_pre)
    print("  [GRU] RE-ANCHOR (kembali pakai Value)")
    gru = train_nn(gru, tr3, va3, epochs=EPOCHS_REANCHOR, lr=LR, name="GRU-REANCHOR")

    # ===== Evaluasi (semua DI DALAM LOOP) =====
    # a) Baseline klasik
    mae, mse, r2 = evaluate(y_te_cls, yhat_lin_te)
    rows.append({"horizon": target_col, "stage": "Linear baseline", "MAE": mae, "MSE": mse, "R2": r2})
    mae, mse, r2 = evaluate(y_te_cls, yhat_combo_te)
    rows.append({"horizon": target_col, "stage": "Linear+RF residual", "MAE": mae, "MSE": mse, "R2": r2})

    # b) GRU final dengan INPUT = VALUE (test PRETRAIN/RE-ANCHOR)
    y_s_true_val, y_s_pred_val = predict_nn(gru, te3)
    y_true_val = sy_pre.inverse_transform(y_s_true_val.reshape(-1,1)).ravel()
    y_pred_val = sy_pre.inverse_transform(y_s_pred_val.reshape(-1,1)).ravel()
    mae, mse, r2 = evaluate(y_true_val, y_pred_val)
    rows.append({"horizon": target_col, "stage": "GRU(final, input=VALUE)", "MAE": mae, "MSE": mse, "R2": r2})

    # c) GRU final dengan INPUT = COMBO (test FINETUNE) ← yang kamu minta
    y_s_true_combo, y_s_pred_combo = predict_nn(gru, te2)
    y_true_combo = sy_combo.inverse_transform(y_s_true_combo.reshape(-1,1)).ravel()
    y_pred_combo = sy_combo.inverse_transform(y_s_pred_combo.reshape(-1,1)).ravel()
    mae, mse, r2 = evaluate(y_true_combo, y_pred_combo)
    rows.append({"horizon": target_col, "stage": "GRU(final, input=COMBO)", "MAE": mae, "MSE": mse, "R2": r2})

# ===== RINGKAS HASIL & SIMPAN =====
results = pd.DataFrame(rows).sort_values(["horizon","stage"]).reset_index(drop=True)
print("\n===== HASIL RINGKAS =====")
print(results)

if SAVE_CSV:
    results.to_csv(CSV_OUT, index=False)
    print(f"\n✅ Tabel metrik disimpan ke: {CSV_OUT}")


/tmp/ipykernel_432/2083771729.py:66: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_numeric(df[c], errors="ignore")



================= Horizon: y_step_3m =================
  [GRU] PRETRAIN (Value-as-Value, slots=Value)
[GRU-PRE] Epoch 01 | val MSE=0.0002
[GRU-PRE] Epoch 05 | val MSE=0.0002
[GRU-PRE] Epoch 10 | val MSE=0.0002
[GRU-PRE] Epoch 15 | val MSE=0.0001
  [GRU] FINETUNE (Value=Combo, slots=lin/combo)
[GRU-FT-COMBO] Epoch 01 | val MSE=0.0002
[GRU-FT-COMBO] Epoch 05 | val MSE=0.0002
[GRU-FT-COMBO] Epoch 10 | val MSE=0.0002
[GRU-FT-COMBO] Epoch 15 | val MSE=0.0002
  [GRU] RE-ANCHOR (kembali pakai Value)
[GRU-REANCHOR] Epoch 01 | val MSE=0.0001
[GRU-REANCHOR] Epoch 05 | val MSE=0.0001
[GRU-REANCHOR] Epoch 10 | val MSE=0.0001

================= Horizon: y_step_6m =================
  [GRU] PRETRAIN (Value-as-Value, slots=Value)
[GRU-PRE] Epoch 01 | val MSE=0.0003
[GRU-PRE] Epoch 05 | val MSE=0.0003
[GRU-PRE] Epoch 10 | val MSE=0.0002
[GRU-PRE] Epoch 15 | val MSE=0.0002
  [GRU] FINETUNE (Value=Combo, slots=lin/combo)
[GRU-FT-COMBO] Epoch 01 | val MSE=0.0005
[GRU-FT-COMBO] Epoch 05 | val MSE=0.0002
[

In [3]:
# ============================================================
# FULL PIPELINE + SAVE MODELS & METRICS (ALL MODELS)
# - Baseline: Linear, Linear+RF residual (Combo)
# - Deep: GRU & LSTM (3 Tahap: PRETRAIN(Value) -> FINETUNE(Combo) -> RE-ANCHOR(Value))
# - Simpan:
#     * Metrik test semua model (CSV gabungan)
#     * Prediksi test per horizon (CSV per-horizon)
#     * Model: linear.joblib, rf.joblib, gru.pt, lstm.pt
#     * Scalers deep: sy_pre.joblib (VALUE), sy_combo.joblib (COMBO)
# ============================================================

from pathlib import Path
import re, os, json, datetime
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ----------------- EDIT DI SINI -----------------
DATA_PATH   = r"/workspace/Machine_Value_Processed.xlsx"
SHEET_NAME  = "SW_3min"
IS_EXCEL    = True

HORIZONS    = ["y_step_3m","y_step_6m","y_step_12m","y_step_24m","y_step_48m"]
MAX_LAG     = 3
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# RNN settings
SEQ_LEN          = 32
BATCH            = 64
EPOCHS_PRETRAIN  = 25
EPOCHS_FINETUNE  = 15
EPOCHS_REANCHOR  = 10
LR               = 1e-3
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
SEED             = 42

# OUTPUTS
RUN_NAME  = f"run_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUT_DIR   = Path(f"outputs_{RUN_NAME}")
SAVE_PRED = True
# ------------------------------------------------

# ============== Utils ==============
def set_seed(s=42):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)

def time_split_idx(n, val_ratio=0.15, test_ratio=0.15):
    n_test = int(n*test_ratio); n_val = int(n*val_ratio)
    n_train = n - n_val - n_test
    assert min(n_train, n_val, n_test) > 0, "Dataset terlalu kecil."
    return n_train, n_train+n_val

def evaluate(y_true, y_pred):
    return dict(
        MAE = float(mean_absolute_error(y_true, y_pred)),
        MSE = float(mean_squared_error(y_true, y_pred)),
        R2  = float(r2_score(y_true, y_pred)),
    )

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def save_sklearn(model, path: Path):
    ensure_dir(path.parent); joblib.dump(model, path)

def save_json(obj, path: Path):
    ensure_dir(path.parent); path.write_text(json.dumps(obj, indent=2))

def save_torch_state(model, path: Path, meta: dict = None):
    ensure_dir(path.parent)
    torch.save({"state_dict": model.state_dict(), "meta": meta or {}}, path)

# ============== Load data ==============
p = Path(DATA_PATH); assert p.exists(), f"Tidak ditemukan: {DATA_PATH}"
df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME) if IS_EXCEL else pd.read_csv(DATA_PATH)
for c in df.columns:
    if c != "TagNumber":
        df[c] = pd.to_numeric(df[c], errors="ignore")

lag_cols_all = [c for c in df.columns if re.fullmatch(r"y_lag_\d+", c)]
lag_idx = sorted(int(c.split("_")[-1]) for c in lag_cols_all)
assert lag_idx, "Kolom y_lag_* tidak ditemukan."
use_lags = [f"y_lag_{k}" for k in lag_idx if k <= MAX_LAG]
assert len(use_lags) >= 1, "Kolom lag sesuai MAX_LAG tidak tersedia."

# ============== Datasets & Models ==============
class SeqDS(Dataset):
    def __init__(self, X, y): self.X=torch.from_numpy(X); self.y=torch.from_numpy(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class GRUReg(nn.Module):
    def __init__(self, in_dim=6, hid=128, layers=1, dropout=0.5):
        super().__init__()
        self.gru = nn.GRU(in_dim, hid, num_layers=layers, batch_first=True,
                          dropout=dropout if layers>1 else 0.0)
        self.fc  = nn.Linear(hid, 1)
    def forward(self, x):
        out,_ = self.gru(x); z = out[:, -1, :]
        return self.fc(z).squeeze(-1)

class LSTMReg(nn.Module):
    def __init__(self, in_dim=6, hid=128, layers=1, dropout=0.5):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hid, num_layers=layers, batch_first=True,
                            dropout=dropout if layers>1 else 0.0)
        self.fc   = nn.Linear(hid, 1)
    def forward(self, x):
        out,_ = self.lstm(x); z = out[:, -1, :]
        return self.fc(z).squeeze(-1)

def train_nn(model, tr_dl, va_dl, epochs=10, lr=1e-3, name="RNN"):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    loss = nn.MSELoss()
    best = {"mse": float("inf"), "state": None}
    for ep in range(1, epochs+1):
        model.train()
        for xb,yb in tr_dl:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); pred = model(xb); l=loss(pred,yb)
            l.backward(); opt.step()
        # quick val
        model.eval(); totv=0.0
        with torch.no_grad():
            for xb,yb in va_dl:
                xb,yb = xb.to(DEVICE), yb.to(DEVICE)
                totv += loss(model(xb), yb).item()*len(yb)
        val_mse = totv/len(va_dl.dataset)
        if val_mse < best["mse"]:
            best["mse"]=val_mse; best["state"]={k:v.cpu().clone() for k,v in model.state_dict().items()}
        if ep % 5 == 0 or ep == 1:
            print(f"[{name}] Epoch {ep:02d} | val MSE={val_mse:.4f}")
    model.load_state_dict(best["state"])
    return model

def predict_nn(model, dl):
    model.eval(); preds=[]; tgts=[]
    with torch.no_grad():
        for xb,yb in dl:
            pr = model(xb.to(DEVICE)).cpu().numpy()
            preds.append(pr); tgts.append(yb.numpy())
    return np.concatenate(tgts), np.concatenate(preds)

# ============== Classical: Linear + RF residual ==============
def classic_fit_predict_full(df, lag_cols, target_col, val_ratio, test_ratio):
    d = df[lag_cols + [target_col]].dropna().reset_index(drop=True)
    X = d[lag_cols].values.astype(float)
    y = d[target_col].values.astype(float)
    n_train, n_trainval = time_split_idx(len(y), val_ratio, test_ratio)
    X_tr, y_tr = X[:n_train], y[:n_train]
    X_va, y_va = X[n_train:n_trainval], y[n_train:n_trainval]
    X_te, y_te = X[n_trainval:], y[n_trainval:]

    lin = LinearRegression().fit(X_tr, y_tr)
    yhat_tr = lin.predict(X_tr); yhat_va = lin.predict(X_va); yhat_te = lin.predict(X_te)

    res_tr  = y_tr - yhat_tr
    rf = RandomForestRegressor(n_estimators=400, random_state=SEED, n_jobs=-1).fit(X_tr, res_tr)
    rhat_tr = rf.predict(X_tr); rhat_va = rf.predict(X_va); rhat_te = rf.predict(X_te)

    lin_full   = np.concatenate([yhat_tr, yhat_va, yhat_te])
    combo_full = np.concatenate([yhat_tr + rhat_tr, yhat_va + rhat_va, yhat_te + rhat_te])
    return d, lin, rf, (X_tr, X_va, X_te, y_tr, y_va, y_te), lin_full, combo_full, (yhat_te, yhat_te + rhat_te)

# ============== Builders sekuens untuk deep ==============
def build_seq_VALUE(df, classic_df, lag_cols, target_col, seq_len=32):
    # [lag1, lag2, lag3, value_norm, value_slot, value_slot]
    tmp = df[["Value"] + lag_cols + [target_col]].dropna().reset_index()
    N = len(classic_df); tmp = tmp.iloc[:N].copy().reset_index(drop=True)
    val = tmp["Value"].astype(float).values
    lags = tmp[lag_cols].values.astype(float)
    tgt  = tmp[target_col].astype(float).values

    sx = StandardScaler().fit(np.column_stack([lags, val]))
    sy = StandardScaler().fit(tgt.reshape(-1,1))
    base = sx.transform(np.column_stack([lags, val]))
    l1, l2, l3 = base[:,0], base[:,1], base[:,2]
    v_norm     = base[:,3]
    v_slot     = sy.transform(val.reshape(-1,1)).ravel()

    y_s = sy.transform(tgt.reshape(-1,1)).ravel()
    F = np.column_stack([l1, l2, l3, v_norm, v_slot, v_slot]).astype(np.float32)

    Xs, Ys = [], []
    for i in range(len(y_s) - seq_len):
        Xs.append(F[i:i+seq_len]); Ys.append(y_s[i+seq_len])
    return np.array(Xs, np.float32), np.array(Ys, np.float32), sy

def build_seq_COMBO(df, classic_df, lag_cols, target_col, lin_full, combo_full, seq_len=32):
    # [lag1, lag2, lag3, combo_as_value, lin_pred, combo_pred]
    tmp = df[["Value"] + lag_cols + [target_col]].dropna().reset_index()
    N = len(classic_df); tmp = tmp.iloc[:N].copy().reset_index(drop=True)
    val = tmp["Value"].astype(float).values
    lags = tmp[lag_cols].values.astype(float)
    tgt  = tmp[target_col].astype(float).values

    sx = StandardScaler().fit(np.column_stack([lags, val]))
    sy = StandardScaler().fit(tgt.reshape(-1,1))
    base = sx.transform(np.column_stack([lags, val]))
    l1, l2, l3 = base[:,0], base[:,1], base[:,2]

    lin_s   = sy.transform(lin_full.reshape(-1,1)).ravel()
    combo_s = sy.transform(combo_full.reshape(-1,1)).ravel()

    y_s = sy.transform(tgt.reshape(-1,1)).ravel()
    F = np.column_stack([l1, l2, l3, combo_s, lin_s, combo_s]).astype(np.float32)

    Xs, Ys = [], []
    for i in range(len(y_s) - seq_len):
        Xs.append(F[i:i+seq_len]); Ys.append(y_s[i+seq_len])
    return np.array(Xs, np.float32), np.array(Ys, np.float32), sy

# ============== MAIN ==============
OUT_DIR.mkdir(parents=True, exist_ok=True)
rows = []

for target_col in HORIZONS:
    if target_col not in df.columns:
        print(f"[SKIP] {target_col} tidak ada."); continue
    print(f"\n================= Horizon: {target_col} =================")

    # ----- Baseline klasik -----
    (classic_df, lin_model, rf_model,
     (X_tr, X_va, X_te, y_tr, y_va, y_te),
     lin_full, combo_full, (yhat_lin_te, yhat_combo_te)) = classic_fit_predict_full(
        df, use_lags, target_col, VAL_RATIO, TEST_RATIO
    )

    # Save classical models
    hz_dir = OUT_DIR / target_col
    ensure_dir(hz_dir)
    save_sklearn(lin_model, hz_dir / f"{target_col}_linear.joblib")
    save_sklearn(rf_model,  hz_dir / f"{target_col}_rf_residual.joblib")
    # Save simple config/info
    save_json({"use_lags": use_lags, "val_ratio": VAL_RATIO, "test_ratio": TEST_RATIO}, hz_dir / f"{target_col}_config.json")

    # Classical metrics (test)
    m_lin   = evaluate(y_te, yhat_lin_te)
    m_combo = evaluate(y_te, yhat_combo_te)
    rows.append({"horizon": target_col, "model": "Linear",             **m_lin})
    rows.append({"horizon": target_col, "model": "Linear+RF residual", **m_combo})

    # (Optional) save predictions test (classical)
    if SAVE_PRED:
        pd.DataFrame({
            "y_true": y_te,
            "y_pred_linear": yhat_lin_te,
            "y_pred_combo":  yhat_combo_te,
        }).to_csv(hz_dir / f"{target_col}_test_preds_classical.csv", index=False)

    # ----- Build deep datasets (VALUE & COMBO) -----
    X_val_pre, y_val_pre, sy_pre   = build_seq_VALUE(df, classic_df, use_lags, target_col, SEQ_LEN)
    X_combo,   y_combo,   sy_combo = build_seq_COMBO(df, classic_df, use_lags, target_col, lin_full, combo_full, SEQ_LEN)

    def make_loaders(X, y):
        n_tr, n_trva = time_split_idx(len(y), VAL_RATIO, TEST_RATIO)
        Xtr, ytr = X[:n_tr], y[:n_tr]
        Xva, yva = X[n_tr:n_trva], y[n_tr:n_trva]
        Xte, yte = X[n_trva:], y[n_trva:]
        return (DataLoader(SeqDS(Xtr, ytr), batch_size=BATCH, shuffle=True),
                DataLoader(SeqDS(Xva, yva), batch_size=BATCH, shuffle=False),
                DataLoader(SeqDS(Xte, yte), batch_size=BATCH, shuffle=False))

    tr_val, va_val, te_val = make_loaders(X_val_pre, y_val_pre)
    tr_com, va_com, te_com = make_loaders(X_combo,   y_combo)

    # Save scalers untuk inference inverse-transform
    save_sklearn(sy_pre,   hz_dir / f"{target_col}_sy_pre.joblib")
    save_sklearn(sy_combo, hz_dir / f"{target_col}_sy_combo.joblib")

    # ----- GRU (3 tahap) -----
    gru = GRUReg(in_dim=6, hid=128, layers=1, dropout=0.5)
    gru = train_nn(gru, tr_val, va_val, epochs=EPOCHS_PRETRAIN, lr=LR, name="GRU-PRE")
    gru = train_nn(gru, tr_com, va_com, epochs=EPOCHS_FINETUNE, lr=LR, name="GRU-FT-COMBO")
    gru = train_nn(gru, tr_val, va_val, epochs=EPOCHS_REANCHOR, lr=LR, name="GRU-REANCHOR")

    # Eval GRU (final weights) – VALUE & COMBO inputs
    yv_t, yv_p = predict_nn(gru, te_val); yv_t = sy_pre.inverse_transform(yv_t.reshape(-1,1)).ravel();  yv_p = sy_pre.inverse_transform(yv_p.reshape(-1,1)).ravel()
    yc_t, yc_p = predict_nn(gru, te_com); yc_t = sy_combo.inverse_transform(yc_t.reshape(-1,1)).ravel(); yc_p = sy_combo.inverse_transform(yc_p.reshape(-1,1)).ravel()
    m_gru_val  = evaluate(yv_t, yv_p)
    m_gru_com  = evaluate(yc_t, yc_p)
    rows.append({"horizon": target_col, "model": "GRU(final, input=VALUE)", **m_gru_val})
    rows.append({"horizon": target_col, "model": "GRU(final, input=COMBO)", **m_gru_com})

    # Save GRU weights
    save_torch_state(gru, hz_dir / f"{target_col}_gru.pt",
                     meta={"in_dim": 6, "hid": 128, "layers": 1, "dropout": 0.5,
                           "seq_len": SEQ_LEN, "channels": ["lag1","lag2","lag3","value_or_combo","lin_slot","combo_slot"]})

    # Save deep preds
    if SAVE_PRED:
        pd.DataFrame({"y_true_VALUE": yv_t, "y_pred_GRU_VALUE": yv_p,
                      "y_true_COMBO": yc_t, "y_pred_GRU_COMBO": yc_p}
                    ).to_csv(hz_dir / f"{target_col}_test_preds_gru.csv", index=False)

    # ----- LSTM (3 tahap) -----
    lstm = LSTMReg(in_dim=6, hid=128, layers=1, dropout=0.5)
    lstm = train_nn(lstm, tr_val, va_val, epochs=EPOCHS_PRETRAIN, lr=LR, name="LSTM-PRE")
    lstm = train_nn(lstm, tr_com, va_com, epochs=EPOCHS_FINETUNE, lr=LR, name="LSTM-FT-COMBO")
    lstm = train_nn(lstm, tr_val, va_val, epochs=EPOCHS_REANCHOR, lr=LR, name="LSTM-REANCHOR")

    # Eval LSTM (final weights) – VALUE & COMBO inputs
    yv_t_l, yv_p_l = predict_nn(lstm, te_val); yv_t_l = sy_pre.inverse_transform(yv_t_l.reshape(-1,1)).ravel();  yv_p_l = sy_pre.inverse_transform(yv_p_l.reshape(-1,1)).ravel()
    yc_t_l, yc_p_l = predict_nn(lstm, te_com); yc_t_l = sy_combo.inverse_transform(yc_t_l.reshape(-1,1)).ravel(); yc_p_l = sy_combo.inverse_transform(yc_p_l.reshape(-1,1)).ravel()
    m_lstm_val     = evaluate(yv_t_l, yv_p_l)
    m_lstm_com     = evaluate(yc_t_l, yc_p_l)
    rows.append({"horizon": target_col, "model": "LSTM(final, input=VALUE)", **m_lstm_val})
    rows.append({"horizon": target_col, "model": "LSTM(final, input=COMBO)", **m_lstm_com})

    # Save LSTM weights
    save_torch_state(lstm, hz_dir / f"{target_col}_lstm.pt",
                     meta={"in_dim": 6, "hid": 128, "layers": 1, "dropout": 0.5,
                           "seq_len": SEQ_LEN, "channels": ["lag1","lag2","lag3","value_or_combo","lin_slot","combo_slot"]})

    # Save deep preds
    if SAVE_PRED:
        pd.DataFrame({"y_true_VALUE": yv_t_l, "y_pred_LSTM_VALUE": yv_p_l,
                      "y_true_COMBO": yc_t_l, "y_pred_LSTM_COMBO": yc_p_l}
                    ).to_csv(hz_dir / f"{target_col}_test_preds_lstm.csv", index=False)

# ===== RINGKAS HASIL & SIMPAN =====
results = pd.DataFrame(rows).sort_values(["horizon","model"]).reset_index(drop=True)
results_path = OUT_DIR / "metrics_all_models.csv"
OUT_DIR.mkdir(exist_ok=True, parents=True)
results.to_csv(results_path, index=False)
print("\n===== HASIL RINGKAS (TEST) =====")
print(results)
print(f"\n✅ Semua artefak disimpan di: {OUT_DIR.resolve()}")


/tmp/ipykernel_876/477378211.py:91: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_numeric(df[c], errors="ignore")



================= Horizon: y_step_3m =================
[GRU-PRE] Epoch 01 | val MSE=0.0002
[GRU-PRE] Epoch 05 | val MSE=0.0002
[GRU-PRE] Epoch 10 | val MSE=0.0002
[GRU-PRE] Epoch 15 | val MSE=0.0001
[GRU-PRE] Epoch 20 | val MSE=0.0001
[GRU-PRE] Epoch 25 | val MSE=0.0002
[GRU-FT-COMBO] Epoch 01 | val MSE=0.0002
[GRU-FT-COMBO] Epoch 05 | val MSE=0.0002
[GRU-FT-COMBO] Epoch 10 | val MSE=0.0002
[GRU-FT-COMBO] Epoch 15 | val MSE=0.0002
[GRU-REANCHOR] Epoch 01 | val MSE=0.0001
[GRU-REANCHOR] Epoch 05 | val MSE=0.0001
[GRU-REANCHOR] Epoch 10 | val MSE=0.0002
[LSTM-PRE] Epoch 01 | val MSE=0.0003
[LSTM-PRE] Epoch 05 | val MSE=0.0002
[LSTM-PRE] Epoch 10 | val MSE=0.0002
[LSTM-PRE] Epoch 15 | val MSE=0.0001
[LSTM-PRE] Epoch 20 | val MSE=0.0002
[LSTM-PRE] Epoch 25 | val MSE=0.0001
[LSTM-FT-COMBO] Epoch 01 | val MSE=0.0002
[LSTM-FT-COMBO] Epoch 05 | val MSE=0.0002
[LSTM-FT-COMBO] Epoch 10 | val MSE=0.0002
[LSTM-FT-COMBO] Epoch 15 | val MSE=0.0003
[LSTM-REANCHOR] Epoch 01 | val MSE=0.0001
[LSTM-REA

In [5]:

############################################################
# 4. DEMO PIPELINE END-TO-END
############################################################

# 4a. init NLU multi-head
runner = ForecastMultiRunner(OUT_ROOT)

# 4b. init forecaster utk time-series numeric
# Ganti path ini dengan folder hasil training timeseries kamu
TS_RUN_ROOT = "outputs_20251027_XXXXXX"  # <-- isi sesuai run_name kamu
forecaster = HorizonForecasterClassic(TS_RUN_ROOT)

# 4c. context_state realtime:
#   Harus diisi dengan lag aktual tiap mesin.
#   Di sini aku pakai dummy. Nanti di production:
#   - y_lag_1 = nilai sensor sekarang
#   - y_lag_2 = nilai sensor 1 step sebelumnya (misal 3 menit lalu)
#   - y_lag_3 = nilai sensor 2 step sebelumnya
context_state_example = {
    "y_lag_1": 105.2,
    "y_lag_2": 104.8,
    "y_lag_3": 104.5,
    # kalau modelmu pakai lag lebih banyak, tambahkan di sini
}

examples = [
    "nilai mesin XP888A 2 jam lagi berapa?",
    "coba prediksi semua mesin apakah akan ada anomali 2 jam kedepan?",
    "kapan XP888A bakal turun di bawah 70 derajat? kalau iya jam berapa pertama kalinya?",
    "andaikan data XP888A kemarin jam 10:00 gue ubah jadi 95, efeknya apa 2 jam lagi?",
    "ramalan performa semua mesin untuk 30 menit ke depan, interval 3 menit"
]

print("\n=== DEMO PREDIKSI ===")
for q in examples:
    print("\nQ:", q)

    # (1) NLU heads
    runner_out = runner.predict(q)

    # (2) Normalized spec (fix horizon, scope, intent, etc)
    spec = interpret_query(q, runner_out)

    # (3) Time-series forecasting execution / rollout
    exec_out = execute_spec(spec, forecaster, context_state_example)

    # Print debug
    print("runner_out:")
    print(runner_out)
    print("\nspec:")
    print(spec)
    print("\nexec_out (final answer struct):")
    print(exec_out)
    print("-"*60)


NameError: name 'ForecastMultiRunner' is not defined